## Importing required libraries

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## Transformations to be done on our dataset

In [2]:
import torchvision.transforms as transforms
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

## getting our dataset

In [3]:
from torchvision import datasets

train_data = datasets.FashionMNIST(
    root="train",
    train=True,
    download=True,
    transform=transform
)
test_data = datasets.FashionMNIST(
    root="test",
    train=False,
    download=True,
    transform=transform
)

100%|██████████| 26.4M/26.4M [00:02<00:00, 10.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 160kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.33MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.5MB/s]
100%|██████████| 26.4M/26.4M [00:02<00:00, 10.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 175kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.29MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 23.2MB/s]


In [4]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data,batch_size=32,shuffle=True,num_workers=8)
test_loader = DataLoader(test_data,batch_size=32,shuffle=False,num_workers=8)

len(train_loader),len(test_loader)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


(1875, 313)

## defining the residual block

In [5]:
class ResidualBlock(nn.Module):
  def __init__(self,in_shape,out_shape,stride=1):
    super().__init__()
    self.conv1 = nn.Sequential(
        nn.Conv2d(in_shape,out_shape,kernel_size=3,padding=1,bias=False),
        nn.BatchNorm2d(out_shape),
        nn.ReLU(inplace=True)
    )
    self.conv2 = nn.Sequential(
        nn.Conv2d(out_shape,out_shape,kernel_size=3,padding=1,bias=False),
        nn.BatchNorm2d(out_shape),

    )
    #projection shortcut if there's dimension mismatch
    if in_shape != out_shape:
      self.shortcut = nn.Conv2d(
          in_shape,
          out_shape,
          kernel_size=1,
          bias=False
      )
    else:
        self.shortcut = nn.Identity()

  def forward(self,x:torch.Tensor):
    out = self.conv2(self.conv1(x))
    out += self.shortcut(x)
    return F.relu(out)

## Implementing the Resnet Model

In [6]:
class Resnet(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    ## input layer
    self.conv1 = nn.Sequential(
        nn.Conv2d(3,64,kernel_size=3,stride=1,padding=1,bias=False),
        nn.BatchNorm2d(64),
        nn.ReLU(inplace=True),

    )

    ##stage 1:  3 residual blocks
    self.stage = nn.Sequential(
        ResidualBlock(64,64),
        ResidualBlock(64,64),
        ResidualBlock(64,64)
    )

    ##stage 2 : 4 residual blocks
    self.stage2 = nn.Sequential(
        ResidualBlock(64,128,stride=2),
        ResidualBlock(128,128),
        ResidualBlock(128,128),
        ResidualBlock(128,128)
    )

    ##stage3 : 6 residual blocks
    self.stage3 = nn.Sequential(
        ResidualBlock(128,256,stride=2),
        ResidualBlock(256,256),
        ResidualBlock(256,256),
        ResidualBlock(256,256),
        ResidualBlock(256,256),
        ResidualBlock(256,256)
    )

    ##stage 4 : 3 residual block
    self.stage4 = nn.Sequential(
        ResidualBlock(256,512,stride=2),
         ResidualBlock(512,512),
         ResidualBlock(512,512),


    )




    ## final output
    self.avgpool = nn.AdaptiveAvgPool2d((1,1))
    self.fc = nn.Linear(512,num_classes)


  def forward(self,x:torch.Tensor):
   x = self.stage4(self.stage3(self.stage2(self.stage(self.conv1(x)))))
   ##final out
   x_avg = self.avgpool(x)
   x_avg = torch.flatten(x_avg,1)
   x_final = self.fc(x_avg)

   return x_final

## Performing model info check

In [7]:
model = Resnet(10)
model

Resnet(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (stage): Sequential(
    (0): ResidualBlock(
      (conv1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
      )
      (conv2): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (shortcut): Identity()
    )
    (1): ResidualBlock(
      (conv1): Sequential(
        (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, t

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [9]:
##checking whether the model is in the CPU or GPU
next(model.parameters()).device

device(type='cpu')

## Moving data and model to the GPU

In [10]:
model = model.to(device)
next(model.parameters()).device

device(type='cuda', index=0)

In [11]:
##loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01)

## Training function

In [12]:
def train_step(model: torch.nn.Module,
               dataloader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,device: torch.device):
    # Put model in train mode
    model.train()

    # Setup train loss and train accuracy values
    train_loss, train_acc = 0, 0

    # Loop through data loader data batches
    for batch, (X, y) in enumerate(dataloader):
        # Send data to target device
        X, y = X.to(device), y.to(device)

        # 1. Forward pass
        y_pred = model(X)

        # 2. Calculate  and accumulate loss
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()

        # 3. Optimizer zero grad
        optimizer.zero_grad()

        # 4. Loss backward
        loss.backward()

        # 5. Optimizer step
        optimizer.step()

        # Calculate and accumulate accuracy metrics across all batches
        y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
        train_acc += (y_pred_class == y).sum().item()/len(y_pred)

    # Adjust metrics to get average loss and accuracy per batch
    train_loss = train_loss / len(dataloader)
    train_acc = train_acc / len(dataloader)
    return train_loss, train_acc

In [13]:
def test_step(model: torch.nn.Module,
              dataloader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,device: torch.device):
    # Put model in eval mode
    model.eval()

    # Setup test loss and test accuracy values
    test_loss, test_acc = 0, 0

    # Turn on inference context manager
    with torch.inference_mode():
        # Loop through DataLoader batches
        for batch, (X, y) in enumerate(dataloader):
            # Send data to target device
            X, y = X.to(device), y.to(device)

            # 1. Forward pass
            test_pred_logits = model(X)

            # 2. Calculate and accumulate loss
            loss = loss_fn(test_pred_logits, y)
            test_loss += loss.item()

            # Calculate and accumulate accuracy
            test_pred_labels = test_pred_logits.argmax(dim=1)
            test_acc += ((test_pred_labels == y).sum().item()/len(test_pred_labels))

    # Adjust metrics to get average loss and accuracy per batch
    test_loss = test_loss / len(dataloader)
    test_acc = test_acc / len(dataloader)
    return test_loss, test_acc

In [14]:
from tqdm.auto import tqdm

# 1. Take in various parameters required for training and test steps
def train(model: torch.nn.Module,
          train_dataloader: torch.utils.data.DataLoader,
          test_dataloader: torch.utils.data.DataLoader,
          optimizer: torch.optim.Optimizer,
          loss_fn: torch.nn.Module = nn.CrossEntropyLoss(),
          epochs: int = 5,
          device: torch.device = 'cpu'):

    # 2. Create empty results dictionary
    results = {"train_loss": [],
        "train_acc": [],
        "test_loss": [],
        "test_acc": []
    }

    # 3. Loop through training and testing steps for a number of epochs
    for epoch in tqdm(range(epochs)):
        train_loss, train_acc = train_step(model=model,
                                           dataloader=train_dataloader,
                                           loss_fn=loss_fn,
                                           optimizer=optimizer,
                                           device=device) # Pass device here
        test_loss, test_acc = test_step(model=model,
            dataloader=test_dataloader,
            loss_fn=loss_fn,
            device=device) # Pass device here

        # 4. Print out what's happening
        print(
            f"Epoch: {epoch+1} | "
            f"train_loss: {train_loss:.4f} | "
            f"train_acc: {train_acc:.4f} | "
            f"test_loss: {test_loss:.4f} | "
            f"test_acc: {test_acc:.4f}"
        )

        # 5. Update results dictionary
        # Ensure all data is moved to CPU and converted to float for storage
        results["train_loss"].append(train_loss.item() if isinstance(train_loss, torch.Tensor) else train_loss)
        results["train_acc"].append(train_acc.item() if isinstance(train_acc, torch.Tensor) else train_acc)
        results["test_loss"].append(test_loss.item() if isinstance(test_loss, torch.Tensor) else test_loss)
        results["test_acc"].append(test_acc.item() if isinstance(test_acc, torch.Tensor) else test_acc)

    # 6. Return the filled results at the end of the epochs
    return results

In [15]:
# Set random seeds
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Set number of epochs
NUM_EPOCHS = 5

# Setup loss function and optimizer
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Train model_0
model_0_results = train(model=model,
                        train_dataloader=train_loader,
                        test_dataloader=test_loader,
                        optimizer=optimizer,
                        loss_fn=loss_fn,
                        epochs=NUM_EPOCHS,
                        device=device)

# End the timer and print out how long it took
end_time = timer()
print(f"Total training time: {end_time-start_time:.3f} seconds")

  0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch: 1 | train_loss: 0.6062 | train_acc: 0.7775 | test_loss: 0.5099 | test_acc: 0.8196
Epoch: 2 | train_loss: 0.3680 | train_acc: 0.8665 | test_loss: 0.3073 | test_acc: 0.8906
Epoch: 3 | train_loss: 0.3057 | train_acc: 0.8913 | test_loss: 0.2875 | test_acc: 0.8986
Epoch: 4 | train_loss: 0.2747 | train_acc: 0.9016 | test_loss: 0.2915 | test_acc: 0.9004
Epoch: 5 | train_loss: 0.2463 | train_acc: 0.9109 | test_loss: 0.2816 | test_acc: 0.9003
Total training time: 3689.858 seconds
